# 00 – Data Exploration and Preparation

**QM 640 Data Analytics Capstone – Abdhul Ahadh**

This notebook covers:
- Loading and inspecting the AIID dataset
- Auditing fill rates for all variables
- Analysing the missing data mechanism for CSETv1 fields (MNAR test)
- Engineering the binary `Deployer_Type` feature
- Applying rule-based sector imputation for known deployers (Tier 3 strategy)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_FILE = Path('../data/AIID_Excel_Export-20261005.xlsx')
FIGURES_DIR = Path('../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the Incidents sheet
df = pd.read_excel(DATA_FILE, sheet_name='Incidents')
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head(3)

In [ ]:
# Fill rate audit
fill_rates = (df.notna().sum() / len(df) * 100).sort_values(ascending=False)
print('Fill rates (%) – top 20 columns:')
print(fill_rates.head(20).round(1).to_string())

In [ ]:
# Identify the MIT taxonomy columns vs CSETv1 columns based on fill rate
high_fill  = fill_rates[fill_rates >= 80].index.tolist()
low_fill   = fill_rates[(fill_rates < 20) & (fill_rates > 0)].index.tolist()

print('High-fill columns (>=80%):', high_fill)
print('Low-fill columns (0-20%):', low_fill)

In [ ]:
# --- MNAR Mechanism Test ---
# Dependent variable: is 'Sector of Deployment' missing?
# Predictors: MIT taxonomy fields + year
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Identify the correct column name for Sector of Deployment
sector_col = [c for c in df.columns if 'sector' in c.lower() or 'Sector' in c]
print('Sector column candidate(s):', sector_col)

# Find MIT taxonomy columns (Risk Domain, Intent, etc.)
mit_cols = [c for c in df.columns if any(k in c.lower() for k in ['risk domain','intent','timing','responsible'])]

In [ ]:
# Engineer the 'Deployer_Type' binary variable
# Map well-known enterprise entities to 'enterprise', everything else to 'non-enterprise'

ENTERPRISE_KEYWORDS = [
    'google', 'meta', 'facebook', 'amazon', 'microsoft', 'apple', 'tesla',
    'uber', 'twitter', 'x corp', 'openai', 'netflix', 'spotify', 'alibaba',
    'bytedance', 'tiktok', 'snapchat', 'linkedin', 'paypal', 'stripe',
    'airbnb', 'lyft', 'walmart', 'target', 'jpmorgan', 'goldman', 'hsbc',
    'bank of america', 'wells fargo', 'unitedhealth', 'anthem', 'cigna',
    'boeing', 'lockheed', 'palantir', 'salesforce', 'oracle', 'sap',
    'ibm', 'intel', 'nvidia', 'qualcomm', 'samsung', 'huawei', 'baidu',
    'tencent', 'weibo', 'didichuxing', 'clearview', 'axon',
]

def classify_deployer(name):
    if pd.isna(name):
        return 'non-enterprise'
    name_lower = str(name).lower()
    for kw in ENTERPRISE_KEYWORDS:
        if kw in name_lower:
            return 'enterprise'
    # Heuristic: if it looks like a company (contains LLC, Inc, Corp, Ltd)
    if any(suffix in name_lower for suffix in [' inc', ' llc', ' ltd', ' corp', ' plc']):
        return 'enterprise'
    return 'non-enterprise'

deployer_col = [c for c in df.columns if 'deployer' in c.lower()][0]
df['Deployer_Type'] = df[deployer_col].apply(classify_deployer)

print(df['Deployer_Type'].value_counts())

In [ ]:
# Save the prepared dataset for use in later notebooks
df.to_pickle('../outputs/incidents_prepared.pkl')
print('Saved prepared dataset to outputs/incidents_prepared.pkl')